# Python 数据分析常见问题与练习


本篇是前七篇的补充练习，围绕课程已涉及的 NumPy、Pandas、Matplotlib 和 RFM 用户分群展开，并非网盘中的独立课程模块。

练习数据为本笔记构造的微型订单表，不是原课程 `sales.xlsx`。题目后的参考写法用于说明思路；所有代码均为 Markdown 示例，没有预先执行或附上模拟运行输出。

建议先完成每节题目，再查看参考写法。重点检查字段类型、数据粒度、边界值和业务口径，而不只关注代码是否能运行。


## 一、练习数据与分析约定

在新的 Notebook 中按顺序复制本篇代码。示例依赖 `pandas`、`numpy`、`matplotlib`；使用项目已有的 uv 环境管理方式准备依赖，绘图前确认本机中文字体。

一行代表一笔订单。分析截止日固定为 `2024-12-31`，金额单位为元；有效订单金额必须大于 1 元。订单号重复时保留第一条，前提是重复行的其他字段完全一致。截止日之后的记录不进入本次分析。

```python
import numpy as np
import pandas as pd

raw = pd.DataFrame({
    '会员ID': ['A', 'A', 'B', 'B', 'B', 'C', 'D', 'E', 'F'],
    '订单号': ['O001', 'O002', 'O003', 'O003', 'O004',
               'O005', 'O006', 'O007', 'O008'],
    '提交日期': ['2024-12-01', '2024-12-25', '2024-10-01',
                 '2024-10-01', '2024-12-20', '2024-06-01',
                 '2024-12-30', '2025-01-02', '错误日期'],
    '订单金额': ['100', '200', '50', '50', '?',
                 '1200', '0.5', '80', '90'],
})
cutoff = pd.Timestamp('2024-12-31')
```

会员和订单标识采用字符串，避免把标识误当成数值做运算，也避免丢失前导零。这一类型选择不影响分组统计。


## 二、练习 1：NumPy 轴与统计

**题目：**两行分别是两名学生，三列分别是三门课程。计算每门课程的平均成绩和每名学生的总成绩。

```python
scores = np.array([[80, 90, 70], [60, 80, 100]])

subject_mean = scores.mean(axis=0)
student_total = scores.sum(axis=1)
```

`axis=0` 消去行这一维，得到三门课程的统计值；`axis=1` 消去列这一维，得到两名学生的统计值。理解“沿哪个轴归约”比死记行列更可靠。

**自行核对：**课程均分应为 70、85、85，两人的总成绩都为 240。这是对给定数据的手工计算，不是实际运行输出。


## 三、练习 2：把字符串转为日期和金额

**题目：**保留 `raw`，创建转换后的副本，找出无法解析的日期和金额。

```python
typed = raw.copy()
typed['提交日期'] = pd.to_datetime(typed['提交日期'], errors='coerce')
typed['订单金额'] = pd.to_numeric(typed['订单金额'], errors='coerce')

bad_dates = typed.loc[typed['提交日期'].isna()]
bad_amounts = typed.loc[typed['订单金额'].isna()]
```

`errors='coerce'` 将无法转换的内容标记为缺失：日期为 `NaT`，数值通常为 `NaN`。它不会自动修复数据，应检查这些行并选择处理策略。

本例应识别到 `O008` 的错误日期和 `O004` 的缺失金额。金额缺失可能直接改变用户的 M 值，不宜在没有依据时填成零。


## 四、练习 3：建立清洗记录

**题目：**依次处理关键字段缺失、重复订单、无效金额和未来日期，记录每一步的行数变化。

```python
orders = typed.copy()
audit = []

def record(step, old_count, frame):
    audit.append({
        '步骤': step,
        '处理前': old_count,
        '处理后': len(frame),
        '移除行数': old_count - len(frame),
    })

n = len(orders)
orders = orders.dropna(
    subset=['会员ID', '订单号', '提交日期', '订单金额']
).copy()
record('关键字段缺失', n, orders)

n = len(orders)
orders = orders.drop_duplicates(subset=['订单号']).copy()
record('重复订单', n, orders)

n = len(orders)
orders = orders.loc[orders['订单金额'] > 1].copy()
record('无效金额', n, orders)

n = len(orders)
orders = orders.loc[orders['提交日期'] <= cutoff].copy()
record('晚于截止日', n, orders)

audit_table = pd.DataFrame(audit)
```

**手工核对：**行数依次为 `9 → 7 → 6 → 5 → 4`，最终有 4 笔有效订单、3 个会员。

真实项目中，一个订单号可能对应多条商品明细。这时不能按订单号直接去重，要先确认行粒度，并区分重复记录与合法明细。


## 五、常见问题：`loc` 与 `iloc` 为什么取到不同数据？

标签与位置是两套不同的定位方式，清洗后行索引通常不再连续。

```python
# 前两条记录，按当前位置选择
first_two = orders.iloc[:2]

# 按业务条件选择，并限定返回列
member_a = orders.loc[
    orders['会员ID'].eq('A'),
    ['订单号', '订单金额'],
]

# 按标签选择，标签切片通常包含终点
example = pd.DataFrame({'金额': [10, 20, 30]}, index=[10, 20, 30])
by_label = example.loc[10:20]
by_position = example.iloc[0:2]
```

`iloc` 的位置切片不包含终点；`loc` 的标签切片通常包含终点。不要把清洗后的旧索引当成当前行号。


## 六、练习 4：Series 赋值为什么出现空值？

**题目：**给索引为 10、20 的表添加一列值 1、2，观察直接赋入默认索引的 Series 有什么问题。

```python
frame = pd.DataFrame({'金额': [100, 200]}, index=[10, 20])

# 错误意图：Series 默认索引是 0、1，会按标签对齐
frame['标记'] = pd.Series([1, 2])

# 修正方式一：让 Series 的标签与表一致
frame['标记'] = pd.Series([1, 2], index=frame.index)

# 修正方式二：明确按位置赋值，列表长度须与行数相同
frame['标记'] = [1, 2]
```

Pandas 会按标签对齐 Series，而不是简单按第一个值、第二个值排列。`assign` 接收 Series 时也遵循对齐规则。


## 七、常见问题：修改切片后原表会不会变化？

不要依赖切片和原对象是否共享内存来判断修改结果。Pandas 版本及 Copy-on-Write 行为会影响这类操作，推荐明确表达修改目标。

```python
# 要修改原表：一次 loc 定位并赋值
orders.loc[orders['会员ID'].eq('A'), '备注'] = '重点观察'

# 要独立加工子表：显式复制
member_a = orders.loc[orders['会员ID'].eq('A')].copy()
member_a['备注'] = '单独分析'
```

避免 `df[条件]['列'] = 值` 这类链式赋值。`inplace=True` 也不保证通过子列修改一定作用于父表；接收返回值或对父表使用一次 `loc` 更清晰。


## 八、练习 5：订单数、用户数和非空值数量

**题目：**统计有效订单数、会员数和各会员的订单总额。

```python
order_count = orders['订单号'].nunique()
member_count = orders['会员ID'].nunique()

member_summary = (
    orders.groupby('会员ID', as_index=False)
    .agg(
        订单数=('订单号', 'nunique'),
        订单总额=('订单金额', 'sum'),
        最后购买=('提交日期', 'max'),
    )
)
```

| 写法 | 实际统计对象 |
| --- | --- |
| `len(df)` | 行数 |
| `groupby(...).size()` | 每组行数，包含缺失字段的行 |
| `groupby(...)[列].count()` | 每组该列的非缺失值数量 |
| `列.nunique()` | 不同值的数量，默认不计缺失值 |

本例去重后的订单表可以用行数核对订单数；真实订单明细表需谨慎使用 `count`，金额也可能需要先按订单汇总。


## 九、练习 6：筛选行与筛选组

**题目：**找出单笔金额大于 150 的订单，以及订单总额大于 150 的会员全部订单。两种结果有何不同？

```python
large_orders = orders.loc[orders['订单金额'] > 150]

valuable_members = orders.groupby('会员ID').filter(
    lambda group: group['订单金额'].sum() > 150
)
```

第一种只返回 A 的 200 元订单和 C 的 1200 元订单；第二种保留 A 的两笔订单以及 C 的订单。这是基于给定数据推导的核对点。

`filter` 的条件作用于整个分组，满足条件后保留组内记录；布尔索引直接判断每一行。


## 十、练习 7：检查连接是否放大订单

**题目：**给有效订单添加会员等级，确保每笔订单只匹配到一个等级。

```python
members = pd.DataFrame({
    '会员ID': ['A', 'B', 'C'],
    '会员等级': [3, 1, 5],
})

joined = orders.merge(
    members,
    on='会员ID',
    how='left',
    validate='many_to_one',
    indicator=True,
)
unmatched = joined.loc[joined['_merge'].eq('left_only')]
```

`validate='many_to_one'` 检查右表会员 ID 是否唯一。若右表出现重复键，应先判断它是重复记录还是等级历史，不要随意去重来消除报错。

`indicator=True` 标记连接来源，方便检查未匹配会员。左连接后行数相同只是一个检查点，连接键唯一性和未匹配数量也要确认。


## 十一、练习 8：正确计算 RFM 原始值

**题目：**按本篇固定截止日计算每个会员的 R/F/M，保持一行一个会员。

```python
rfm = (
    orders.groupby('会员ID', as_index=False)
    .agg(
        last_order=('提交日期', 'max'),
        f=('订单号', 'nunique'),
        m=('订单金额', 'sum'),
    )
)
rfm['r'] = (cutoff - rfm['last_order']).dt.days
rfm = rfm[['会员ID', 'r', 'f', 'm']]
```

**手工核对表：**

| 会员 | R：间隔天数 | F：订单数 | M：金额 |
| --- | --- | --- | --- |
| A | 6 | 2 | 300 |
| B | 91 | 1 | 50 |
| C | 213 | 1 | 1200 |

与课程按年度计算的差异是：本练习使用固定截止日，不使用数据中的最大订单日期替代截止日。跨年度分析时还需增加年份分组，并明确每年的参照日期。


## 十二、练习 9：分箱边界与评分方向

**题目：**R 划分为 0–30、31–180、181–365 天；F 划分为 1–2、3–5、6 次及以上；M 划分为不超过 100、100–500、超过 500 元。按 R 越小越好、F/M 越大越好打分。

```python
rfm['r_score'] = pd.cut(
    rfm['r'], bins=[-1, 30, 180, np.inf], labels=[3, 2, 1]
)
rfm['f_score'] = pd.cut(
    rfm['f'], bins=[0, 2, 5, np.inf], labels=[1, 2, 3]
)
rfm['m_score'] = pd.cut(
    rfm['m'], bins=[0, 100, 500, np.inf], labels=[1, 2, 3]
)
```

本练习沿用课程的分箱机制，但使用专门为练习设定的边界，不能当成原课阈值。

> `pd.cut` 默认左开右闭，`bins=[0, 2, 5, ...]` 的第一个区间是 `(0, 2]`，所以 F=2 得 1 分，F=5 得 2 分。不能把边界值说成归入右边的下一个区间。

R 的最后一段在代码中扩展至无穷大，保证以后超过 365 天的记录仍能分箱；本次统计窗口内的 R 最大只有 365 天。


## 十三、常见问题：分箱结果出现 `NaN`

原因包括原始数值缺失、边界未覆盖、数值恰好等于被排除的最小边界等。先查分箱结果，再拼接标签。

```python
score_columns = ['r_score', 'f_score', 'm_score']
bad_scores = rfm.loc[rfm[score_columns].isna().any(axis=1)]
assert bad_scores.empty, '存在没有分箱成功的会员'

rfm['rfm_group'] = (
    rfm['r_score'].astype('string')
    + rfm['f_score'].astype('string')
    + rfm['m_score'].astype('string')
)
```

直接 `astype(str)` 可能将缺失值变成字面量 `'nan'`，使后续 `notna()` 检查看似通过。必须在转换前检查缺失，或使用保留缺失语义的可空字符串类型。

**手工核对：**本篇分组应为 A=`312`、B=`211`、C=`113`。三个标签的相加总分不能代替组合编码，因为相同总分可能对应完全不同的消费特征。


## 十四、练习 10：交叉表和透视表

**题目：**给订单表添加高低金额标签，统计各会员的高低金额订单数量，并比较平均订单金额。

```python
orders['金额类别'] = np.where(orders['订单金额'] > 150, '高', '低')

frequency_table = pd.crosstab(orders['会员ID'], orders['金额类别'])
average_table = orders.pivot_table(
    values='订单金额',
    index='会员ID',
    columns='金额类别',
    aggfunc='mean',
)
```

交叉表默认统计频数，透视表这里统计平均金额。透视表中某类没有订单时，平均值是缺失值，不能自动填零后解释为“平均消费为零”。

若需要组内比例，可以用 `pd.crosstab(..., normalize='index')`，或按行总数对频数表做除法。分母是每个会员的订单数，不是全体订单数。


## 十五、常见问题：累计涨跌幅为什么不等于累计收益？

课程用股票涨跌幅的 `cumsum()` 演示累计运算。它表示百分数的累加，用于学习累计函数；复利收益需要乘法累计。

```python
returns_pct = pd.Series([10.0, -10.0])

# 涨跌百分数直接相加
sum_of_changes = returns_pct.cumsum()

# 每期收益率转换成倍率再累计
compound_return = (1 + returns_pct / 100).cumprod() - 1
```

先涨 10% 再跌 10%，合计涨跌幅为 0，但资产倍率是 `1.1 × 0.9 = 0.99`，累计收益为 -1%。实际分析还应先按日期排序，并确认涨跌幅字段的单位。


## 十六、练习 11：绘图与保存

**题目：**绘制各会员累计金额的柱状图，并保存图片。示例字体以本机安装情况为准。

```python
from pathlib import Path
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

reports = Path('reports')
reports.mkdir(parents=True, exist_ok=True)

amount = rfm.sort_values('m', ascending=False)
fig, ax = plt.subplots(figsize=(8, 5), dpi=120)
ax.bar(amount['会员ID'], amount['m'], color='steelblue')
ax.set_title('会员累计有效订单金额')
ax.set_xlabel('会员ID')
ax.set_ylabel('金额（元）')
ax.grid(axis='y', linestyle='--', alpha=0.3)
fig.tight_layout()
fig.savefig(reports / 'member_amount.png')
plt.show()
```

保存必须发生在显示或关闭画布前。使用明确的 `fig.savefig` 能避免把其他当前图误保存；图表标题应说明这是清洗后的有效订单金额，而不是原始金额。


## 十七、练习 12：导出结果并保留标识

**题目：**导出 RFM 明细，读回检查标识、记录数量和列名。

```python
output_path = reports / 'practice_rfm.csv'
rfm.to_csv(output_path, index=False, encoding='utf-8-sig')

restored = pd.read_csv(
    output_path,
    dtype={'会员ID': 'string', 'rfm_group': 'string'},
)
assert len(restored) == len(rfm)
assert list(restored.columns) == list(rfm.columns)
assert restored['会员ID'].is_unique
assert restored['rfm_group'].str.fullmatch(r'[123]{3}').all()
```

`index=False` 避免把默认索引保存为额外字段。CSV 不记录 Pandas 类型信息，所以读回时显式指定标识列类型。年度表的唯一性应检查 `year + 会员ID`，不能要求会员 ID 在整张年度表中唯一。


## 十八、综合练习与参考检查点

从最初的 `raw` 开始，完成一次完整分析。不要直接从中间变量或上一轮运行结果继续。

1. 转换日期和金额，列出解析失败的订单。
2. 根据本篇业务规则清洗，生成逐步行数记录。
3. 核对最终订单号唯一、金额大于 1、日期不晚于截止日。
4. 按会员汇总 R/F/M，并与手工核对表比较。
5. 分箱前检查范围，分箱后检查缺失和三个评分方向。
6. 生成分群编码、交叉表、金额柱状图和 CSV 文件。

**参考检查点：**清洗后 4 笔订单，3 个会员，累计金额 1550 元；每个会员的指标和编码见第十一至十三节。

这份微型数据只检验计算逻辑，不能用于推断真实用户群比例或提出实际营销结论。换成课程 `sales.xlsx` 时，应恢复原课的年度粒度、截止日期、阈值和字段，并重新检查清洗后的分布。


## 十九、完成后的复盘

| 现象 | 优先检查 |
| --- | --- |
| 金额平均值无法计算 | 数值列是否仍含字符串 |
| 新增列全是缺失值 | Series 索引是否与 DataFrame 对齐 |
| 合并后行数变多 | 连接键是否重复，是否发生多对多匹配 |
| R 值为负数 | 是否包含截止日之后的订单 |
| F 异常偏大 | 数据是否为订单明细，重复订单是否处理 |
| 分箱标签缺失 | 区间覆盖、缺失值和开闭边界 |
| 编码出现 `nan` | 是否在转换字符串前检查分箱结果 |
| 读回后出现 `Unnamed: 0` | 写出时是否包含索引 |

完成练习后，应能从原始记录解释每个汇总指标，并说明每条清洗规则的依据。能够复查结果，比只记住某个函数名更重要。
